# M5 weekly panel for ICDN

Step-by-step construction of the observation unit

`(store, product, week) → (price, units, promo)`

We do **not** call `builder.run()`. Each cell invokes one method so we can inspect the intermediate objects.

**Files used:** `sales_train_evaluation.csv` (preferred), `sell_prices.csv`, `calendar.csv`.  
**Not used:** concatenation with `sales_train_validation.csv`, `sample_submission.csv`.

**ICDN 1.0.0:** `units > 0` is required (`log(units)`). Zeros stay in the master panel and are dropped only at the end, never recoded as ones.

In [21]:
from pathlib import Path
import sys

import pandas as pd

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "src"))
from m5 import M5Config, M5WeeklyPanelBuilder

DATA_DIR = PROJECT_ROOT / "data" / "M5-walmart"
OUT_DIR = DATA_DIR / "panel"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR    :", DATA_DIR)
print("Exists      :", DATA_DIR.exists())
print("Files       :", sorted(p.name for p in DATA_DIR.glob("*.csv")) if DATA_DIR.exists() else "—")

PROJECT_ROOT: /home/thebigmonster/Github/nn-elasticity-additional-work
DATA_DIR    : /home/thebigmonster/Github/nn-elasticity-additional-work/data/M5-walmart
Exists      : True
Files       : ['calendar.csv', 'sales_train_evaluation.csv', 'sales_train_validation.csv', 'sample_submission.csv', 'sell_prices.csv']


## 0. Config and builder

Selection thresholds are stored now but applied only on the **first half** of complete weeks, never on the full sample.

In [22]:
config = M5Config(
    data_dir=DATA_DIR,
    out_dir=OUT_DIR,
    min_positive_rate=0.90,
    min_stores=8,
    min_unique_prices=8,
    min_eligible_in_department=15,
    n_skus=20,
    selection_frac=0.50,
)

builder = M5WeeklyPanelBuilder(config)

## 1. Resolve the sales file

Prefer `sales_train_evaluation.csv`: it already contains `d_1…d_1913` and extends to `d_1941`.

In [23]:
sales_file = builder.resolve_sales_file()

Using: sales_train_evaluation.csv


## 2. Calendar and sequential `week_id`

`wm_yr_wk` codes wrap the year (`11152` → `11201`). ICDN lags treat `week_id` as consecutive integers, so we map:

`date → d → wm_yr_wk → week_id ∈ {1, 2, …, T}`

In [24]:
calendar = builder.load_calendar()
calendar.head()

,date,wm_yr_wk,weekday,wday,month,year,d,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI
0,2011-01-29,11101,Saturday,1,1,2011,d_1,NaN,NaN,NaN,NaN,0,0,0
1,2011-01-30,11101,Sunday,2,1,2011,d_2,NaN,NaN,NaN,NaN,0,0,0
2,2011-01-31,11101,Monday,3,1,2011,d_3,NaN,NaN,NaN,NaN,0,0,0
3,2011-02-01,11101,Tuesday,4,2,2011,d_4,NaN,NaN,NaN,NaN,1,1,0
4,2011-02-02,11101,Wednesday,5,2,2011,d_5,NaN,NaN,NaN,NaN,1,0,1


In [25]:
week_map = builder.build_week_index()
week_map.head(15)

        date     d  wm_yr_wk  week_id
0 2011-01-29   d_1     11101        1
1 2011-01-30   d_2     11101        1
2 2011-01-31   d_3     11101        1
3 2011-02-01   d_4     11101        1
4 2011-02-02   d_5     11101        1
5 2011-02-03   d_6     11101        1
6 2011-02-04   d_7     11101        1
7 2011-02-05   d_8     11102        2
8 2011-02-06   d_9     11102        2
9 2011-02-07  d_10     11102        2


,wm_yr_wk,week_id
0,11101,1
1,11102,2
2,11103,3
3,11104,4
4,11105,5
5,11106,6
6,11107,7
7,11108,8
8,11109,9
9,11110,10


In [26]:
# Year wrap: wm_yr_wk jumps; week_id must not.
wrap = week_map.copy()
wrap["wm_yr_wk_diff"] = wrap["wm_yr_wk"].diff()
wrap["week_id_diff"] = wrap["week_id"].diff()
wrap.loc[wrap["wm_yr_wk_diff"] > 1]

,wm_yr_wk,week_id,wm_yr_wk_diff,week_id_diff
52,11201,53,49.0000,1.0000
104,11301,105,49.0000,1.0000
157,11401,158,48.0000,1.0000
209,11501,210,49.0000,1.0000
261,11601,262,49.0000,1.0000


## 3. Load wide daily sales

Do **not** `melt` all `d_*` columns (~59M rows). We will sum inside each week first.

This cell is the RAM bottleneck.

In [27]:
sales = builder.load_sales()

print("item-store rows:", sales.shape[0])
print("day columns    :", len(builder.day_cols))
print("first/last day :", builder.day_cols[0], builder.day_cols[-1])
sales[builder.META_COLS + builder.day_cols[:3]].head()

(30490, 1947)
1941
item-store rows: 30490
day columns    : 1941
first/last day : d_1 d_1941


,item_id,dept_id,cat_id,store_id,state_id,d_1,d_2,d_3
0,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,0.0000,0.0000,0.0000
1,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,0.0000,0.0000,0.0000
2,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,0.0000,0.0000,0.0000
3,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,0.0000,0.0000,0.0000
4,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,0.0000,0.0000,0.0000


## 4. Keep only days that exist in the sales file

`calendar.csv` also has forecast dates. Then drop truncated weeks (≠ 7 days).

In [28]:
calendar_train = builder.restrict_calendar_to_observed_days()

print("calendar days :", builder.calendar["d"].nunique())
print("train days    :", calendar_train["d"].nunique())
print("sales d_ cols :", len(builder.day_cols))
calendar_train[["date", "d", "wm_yr_wk", "week_id"]].head()

calendar days : 1969
train days    : 1941
sales d_ cols : 1941


,date,d,wm_yr_wk,week_id
0,2011-01-29,d_1,11101,1
1,2011-01-30,d_2,11101,1
2,2011-01-31,d_3,11101,1
3,2011-02-01,d_4,11101,1
4,2011-02-02,d_5,11101,1


In [29]:
full_weeks = builder.identify_complete_weeks()

days_per_week = (
    calendar_train.groupby(["week_id", "wm_yr_wk"], observed=True)["d"]
    .nunique()
    .reset_index(name="n_days")
)
days_per_week.loc[days_per_week["n_days"] != 7]

n_days
2      1
7    277
Name: count, dtype: int64
Complete weeks retained: 277


,week_id,wm_yr_wk,n_days
277,278,11617,2


## 5. Daily → weekly units

$$Q_{isw} = \sum_{d \in w} Q_{isd}$$

Inspect a few `week_*` columns **before** the melt. After this method, `builder.sales` is released.

In [30]:
weekly_sales = builder.aggregate_daily_to_weekly()

print("builder.sales is now:", builder.sales)
print("weekly_sales shape  :", weekly_sales.shape)
print("week_id range       :", weekly_sales["week_id"].min(), "→", weekly_sales["week_id"].max())
weekly_sales.head()

/home/thebigmonster/Github/nn-elasticity-additional-work/src/m5.py:332: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  weekly_sales[col] = (
/home/thebigmonster/Github/nn-elasticity-additional-work/src/m5.py:332: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  weekly_sales[col] = (
/home/thebigmonster/Github/nn-elasticity-additional-work/src/m5.py:332: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at onc

builder.sales is now: None
weekly_sales shape  : (8445730, 7)
week_id range       : 1 → 277


,item_id,dept_id,cat_id,store_id,state_id,units,week_id
0,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,0.0000,1
1,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,0.0000,1
2,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,0.0000,1
3,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,0.0000,1
4,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,0.0000,1


## 6. Attach `wm_yr_wk`

Needed to join `sell_prices.csv`, which is keyed by Walmart week codes, not `week_id`.

In [31]:
weekly_sales = builder.attach_wm_yr_wk(weekly_sales)
weekly_sales.head()

,item_id,dept_id,cat_id,store_id,state_id,units,week_id,wm_yr_wk
0,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,0.0000,1,11101
1,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,0.0000,1,11101
2,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,0.0000,1,11101
3,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,0.0000,1,11101
4,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,0.0000,1,11101


## 7–8. Prices and markdown promo proxy

Missing price ⇒ item not offered that store-week.

Promo is **not** SNAP. It is:

$$P^{ref}_{is,t} = \max\{P_{is,t-13},\ldots,P_{is,t-1}\},\quad
\mathrm{on\_promo}=1[P_{ist} \le 0.95\, P^{ref}_{is,t}]$$

`shift(1)` excludes the current week from the reference.

In [32]:
prices = builder.load_prices()
print(prices.shape)
prices.head()

(6841121, 5)


,store_id,item_id,wm_yr_wk,sell_price,week_id
0,CA_1,HOBBIES_1_001,11325,9.5800,129
1,CA_1,HOBBIES_1_001,11326,9.5800,130
2,CA_1,HOBBIES_1_001,11327,8.2600,131
3,CA_1,HOBBIES_1_001,11328,8.2600,132
4,CA_1,HOBBIES_1_001,11329,8.2600,133


In [33]:
prices = builder.construct_markdown_promo(prices)

print("promo rate:", prices["on_promo"].mean())
print("ref missing (early weeks / short history):", prices["regular_price_ref"].isna().mean())

# One SKU-store to see the proxy in time
example = (
    prices.dropna(subset=["regular_price_ref"])
    .query("on_promo == 1")
    [["store_id", "item_id", "week_id", "wm_yr_wk", "sell_price", "regular_price_ref", "on_promo"]]
    .head(12)
)
example

promo rate: 0.029828006258038706
ref missing (early weeks / short history): 0.01782748762958585


,store_id,item_id,week_id,wm_yr_wk,sell_price,regular_price_ref,on_promo
4,CA_1,HOBBIES_1_001,133,11329,8.2600,9.5800,1
5,CA_1,HOBBIES_1_001,134,11330,8.2600,9.5800,1
6,CA_1,HOBBIES_1_001,135,11331,8.2600,9.5800,1
7,CA_1,HOBBIES_1_001,136,11332,8.2600,9.5800,1
8,CA_1,HOBBIES_1_001,137,11333,8.2600,9.5800,1
9,CA_1,HOBBIES_1_001,138,11334,8.2600,9.5800,1
10,CA_1,HOBBIES_1_001,139,11335,8.2600,9.5800,1
11,CA_1,HOBBIES_1_001,140,11336,8.2600,9.5800,1
12,CA_1,HOBBIES_1_001,141,11337,8.2600,9.5800,1
13,CA_1,HOBBIES_1_001,142,11338,8.2600,9.5800,1


## 9. Merge: price panel is the universe

Left join sales onto prices.  
Price present + sales missing → `units = 0` (offered, not sold).  
No price → row does not enter the master (not in assortment).

In [34]:
master_raw = builder.merge_availability_and_sales(prices, weekly_sales)

print("imputed zeros:", builder.n_imputed_zero_units)
print("shape        :", master_raw.shape)
print("units == 0   :", (master_raw["units"] == 0).mean())
master_raw.head()

Missing units after merge: 0
imputed zeros: 0
shape        : (6688671, 10)
units == 0   : 0.23904225518043867


,store_id,item_id,wm_yr_wk,week_id,sell_price,on_promo,dept_id,cat_id,state_id,units
0,CA_1,HOBBIES_1_001,11325,129,9.5800,0,HOBBIES_1,HOBBIES,CA,1.0000
1,CA_1,HOBBIES_1_001,11326,130,9.5800,0,HOBBIES_1,HOBBIES,CA,0.0000
2,CA_1,HOBBIES_1_001,11327,131,8.2600,0,HOBBIES_1,HOBBIES,CA,2.0000
3,CA_1,HOBBIES_1_001,11328,132,8.2600,0,HOBBIES_1,HOBBIES,CA,2.0000
4,CA_1,HOBBIES_1_001,11329,133,8.2600,1,HOBBIES_1,HOBBIES,CA,6.0000


## 10. ICDN schema

Rename to `store_code`, `product_code`, `price`, `category` (`dept_id`, e.g. `FOODS_3`).  
Then compact `week_id` to `1…T` over **complete** weeks only (still not `wm_yr_wk`).

In [35]:
master = builder.to_icdn_schema(master_raw)

print("week_id range:", master["week_id"].min(), "→", master["week_id"].max())
print("n weeks      :", master["week_id"].nunique())
# Compacted week_id is consecutive
assert (master["week_id"].sort_values().unique() == range(1, master["week_id"].nunique() + 1)).all()
master.head()

week_id range: 1 → 277
n weeks      : 277


,store_code,product_code,week_id,price,units,on_promo,category,cat_id,state_id,wm_yr_wk
0,CA_1,HOBBIES_1_001,129,9.5800,1.0000,0,HOBBIES_1,HOBBIES,CA,11325
1,CA_1,HOBBIES_1_001,130,9.5800,0.0000,0,HOBBIES_1,HOBBIES,CA,11326
2,CA_1,HOBBIES_1_001,131,8.2600,2.0000,0,HOBBIES_1,HOBBIES,CA,11327
3,CA_1,HOBBIES_1_001,132,8.2600,2.0000,0,HOBBIES_1,HOBBIES,CA,11328
4,CA_1,HOBBIES_1_001,133,8.2600,6.0000,1,HOBBIES_1,HOBBIES,CA,11329


## 11. Validate and save the master

The master **keeps zeros**. This is the scientific dataset. Do not delete it.

In [36]:
builder.validate_master(master)
builder.master = builder.save_master(master)
builder.master.head()

Wrote /home/thebigmonster/Github/nn-elasticity-additional-work/data/M5-walmart/panel/m5_weekly_master.parquet


,store_code,product_code,week_id,price,units,on_promo,category,cat_id,state_id,wm_yr_wk
0,CA_1,HOBBIES_1_008,1,0.4600,37.0000,0,HOBBIES_1,HOBBIES,CA,11101
1,CA_1,HOBBIES_1_009,1,1.5600,17.0000,0,HOBBIES_1,HOBBIES,CA,11101
2,CA_1,HOBBIES_1_010,1,3.1700,1.0000,0,HOBBIES_1,HOBBIES,CA,11101
3,CA_1,HOBBIES_1_012,1,5.9800,2.0000,0,HOBBIES_1,HOBBIES,CA,11101
4,CA_1,HOBBIES_1_015,1,0.7000,9.0000,0,HOBBIES_1,HOBBIES,CA,11101


## 12. Sparsity diagnostics (full sample)

Do **not** filter `units > 0` before ranking SKUs. That would hide sparsity.

In [37]:
product_stats = builder.diagnose_sparsity()
product_stats.head(20)

Rows: 6688671
Stores: 10
Products: 3049
Weeks: 277
Zero-sales share: 0.23904225518043867
Promo share: 0.03014126423619879
Imputed zero units: 0
Wrote /home/thebigmonster/Github/nn-elasticity-additional-work/data/M5-walmart/panel/m5_product_diagnostics.csv


,product_code,category,n_obs,positive_obs,total_units,mean_units,n_stores,n_weeks,unique_prices,mean_price,std_price,promo_rate,positive_rate,price_cv
2380,FOODS_3_156,FOODS_3,2614,2614,"43,654.0000",16.7001,10,277,9,3.8668,0.1539,0.0015,1.0000,0.0398
2921,FOODS_3_697,FOODS_3,2704,2704,"100,313.0000",37.0980,10,277,6,3.0414,0.1697,0.0433,1.0000,0.0558
2697,FOODS_3_473,FOODS_3,2615,2615,"99,791.0000",38.1610,10,277,6,3.8686,0.1460,0.0000,1.0000,0.0377
2892,FOODS_3_668,FOODS_3,2770,2770,"138,544.0000",50.0159,10,277,3,1.6045,0.0763,0.0000,1.0000,0.0476
3016,FOODS_3_795,FOODS_3,1395,1395,"29,500.0000",21.1470,10,140,2,2.9748,0.0390,0.0000,1.0000,0.0131
2810,FOODS_3_586,FOODS_3,2770,2769,"931,155.0000",336.1570,10,277,3,1.5944,0.0722,0.0000,0.9996,0.0453
2779,FOODS_3_555,FOODS_3,2770,2769,"497,266.0000",179.5184,10,277,3,1.5944,0.0722,0.0000,0.9996,0.0453
2304,FOODS_3_080,FOODS_3,2770,2769,"265,533.0000",95.8603,10,277,3,1.5944,0.0722,0.0000,0.9996,0.0453
2885,FOODS_3_661,FOODS_3,1081,1080,"28,493.0000",26.3580,10,109,1,3.2800,0.0000,0.0000,0.9991,0.0000
1808,FOODS_1_200,FOODS_1,1007,1006,"37,042.0000",36.7845,10,104,2,0.9723,0.0158,0.0000,0.9990,0.0163


In [38]:
def n_eligible(stats, rate):
    return (
        (stats["positive_rate"] >= rate)
        & (stats["n_stores"] >= config.min_stores)
        & (stats["unique_prices"] >= config.min_unique_prices)
    ).sum()

print("eligible @ 90% (full sample, descriptive only):", n_eligible(product_stats, 0.90))
print("eligible @ 95% (full sample, descriptive only):", n_eligible(product_stats, 0.95))

product_stats.groupby("category", observed=True).size().sort_values(ascending=False)

eligible @ 90% (full sample, descriptive only): 109
eligible @ 95% (full sample, descriptive only): 33


category
FOODS_3        823
HOUSEHOLD_1    532
HOUSEHOLD_2    515
HOBBIES_1      416
FOODS_2        398
FOODS_1        216
HOBBIES_2      149
dtype: int64

## 13. Freeze SKUs without lookahead

Department and 20 SKUs are chosen on `week_id <= cutoff` (first 50%).  
The same list is reused later for OLS, Ridge, MLP and ICDN.

In [40]:
selection = builder.select_products_without_lookahead()
selection.to_dict()

Eligible SKUs by department (selection window):
category
FOODS_3        31
FOODS_2        17
HOBBIES_1      14
HOUSEHOLD_1    13
dtype: int64
Selected department: FOODS_3
Selected SKUs: ['FOODS_3_785', 'FOODS_3_202', 'FOODS_3_136', 'FOODS_3_704', 'FOODS_3_115', 'FOODS_3_324', 'FOODS_3_615', 'FOODS_3_448', 'FOODS_3_098', 'FOODS_3_635', 'FOODS_3_319', 'FOODS_3_541', 'FOODS_3_124', 'FOODS_3_578', 'FOODS_3_808', 'FOODS_3_534', 'FOODS_3_524', 'FOODS_3_404', 'FOODS_3_041', 'FOODS_3_481']
Wrote /home/thebigmonster/Github/nn-elasticity-additional-work/data/M5-walmart/panel/m5_selected_skus.json


{'cutoff_week_id': 139,
 'category': 'FOODS_3',
 'product_codes': ['FOODS_3_785',
  'FOODS_3_202',
  'FOODS_3_136',
  'FOODS_3_704',
  'FOODS_3_115',
  'FOODS_3_324',
  'FOODS_3_615',
  'FOODS_3_448',
  'FOODS_3_098',
  'FOODS_3_635',
  'FOODS_3_319',
  'FOODS_3_541',
  'FOODS_3_124',
  'FOODS_3_578',
  'FOODS_3_808',
  'FOODS_3_534',
  'FOODS_3_524',
  'FOODS_3_404',
  'FOODS_3_041',
  'FOODS_3_481'],
 'n_eligible_in_category': 31,
 'criteria': {'selection_frac': 0.5,
  'min_positive_rate': 0.9,
  'min_stores': 8,
  'min_unique_prices': 8,
  'n_skus': 20}}

## 14. ICDN panel

Full horizon, frozen SKUs, **`units > 0` only**. Zeros are dropped, not replaced by 1.

In [41]:
icdn_panel = builder.build_icdn_panel()

print(icdn_panel.shape)
print(icdn_panel.dtypes)
icdn_panel.head(12)

Zero-sales rows dropped for ICDN: 5431
Wrote /home/thebigmonster/Github/nn-elasticity-additional-work/data/M5-walmart/panel/m5_icdn_panel.parquet
              n_obs  n_weeks  n_stores  mean_units
product_code                                      
FOODS_3_041    2098      225        10     12.5739
FOODS_3_098    2439      272        10      9.0508
FOODS_3_115    2586      277        10     11.4756
FOODS_3_124    2458      269        10     28.4467
FOODS_3_136    2593      275        10     39.4180
FOODS_3_202    2596      277        10    115.5994
FOODS_3_319    2375      261        10     87.7318
FOODS_3_324    2275      274        10     30.1758
FOODS_3_404    2231      265        10     49.3268
FOODS_3_448    2260      272        10     10.4717
FOODS_3_481    2410      277        10     12.5548
FOODS_3_524    2254      262        10     41.8660
FOODS_3_534    2669      277        10     15.0854
FOODS_3_541    2341      262        10    116.4596
FOODS_3_578    2226      258        10

,store_code,product_code,week_id,price,units,on_promo,category
0,CA_1,FOODS_3_098,1,4.3800,9.0000,0,FOODS_3
1,CA_1,FOODS_3_115,1,4.3800,4.0000,0,FOODS_3
2,CA_1,FOODS_3_124,1,5.9800,30.0000,0,FOODS_3
3,CA_1,FOODS_3_136,1,4.3800,20.0000,0,FOODS_3
4,CA_1,FOODS_3_202,1,4.3800,54.0000,0,FOODS_3
5,CA_1,FOODS_3_319,1,1.0000,61.0000,0,FOODS_3
6,CA_1,FOODS_3_324,1,3.0000,62.0000,0,FOODS_3
7,CA_1,FOODS_3_404,1,1.0000,31.0000,0,FOODS_3
8,CA_1,FOODS_3_448,1,3.0000,15.0000,0,FOODS_3
9,CA_1,FOODS_3_481,1,3.5000,36.0000,0,FOODS_3


In [42]:
print("master zeros kept :", (builder.master["units"] == 0).any())
print("icdn  zeros left  :", (icdn_panel["units"] == 0).any())
print("icdn units min    :", icdn_panel["units"].min())
print("unique week_id    :", icdn_panel["week_id"].nunique(), "from", icdn_panel["week_id"].min(), "to", icdn_panel["week_id"].max())
print("outputs:", sorted(p.name for p in OUT_DIR.iterdir()))

master zeros kept : True
icdn  zeros left  : False
icdn units min    : 1.0
unique week_id    : 277 from 1 to 277
outputs: ['m5_icdn_panel.parquet', 'm5_product_diagnostics.csv', 'm5_selected_skus.json', 'm5_selection_window_diagnostics.csv', 'm5_weekly_master.parquet']
